# URL Classification with LightGBM — Amazon SageMaker AI JumpStart
## Phishing Detection Architecture

- URL classification is a structured feature problem
- The LightGBM Model will be trained on 43 URL features (entropy, subdomains, keywords, etc.)
- 651k Kaggle Malicious URL dataset - https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset


### Steps

* 1. Kaggle Dataset Feature extraction then stored in S3
* 2. Amazon SageMaker JumpStart Training Job (lightgbm-classification-model)
* 3. Deployed on Amazon SageMaker AI Real-Time Endpoint
* 4. Evaluation

In [ ]:
!pip install lightgbm tldextract kaggle kagglehub --quiet
print('Packages ready.')

In [ ]:
import os, json, time, warnings
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import kagglehub
from datetime import datetime
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, SourceCode, InputData, StoppingCondition
from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.core.image_uris import retrieve as retrieve_image_uri
from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant
from feature_utils import extract_url_features, FEATURE_COLS_ORDER

tqdm.pandas()
warnings.filterwarnings('ignore')

sess        = Session()
role        = get_execution_role()
region      = boto3.Session().region_name
bucket      = sess.default_bucket()
TIMESTAMP   = datetime.now().strftime('%Y%m%d-%H%M%S')
S3_PREFIX   = f'phishing-lgbm/{TIMESTAMP}'
RANDOM_SEED = 42

SAMPLE_SIZE = None

os.makedirs('./data',   exist_ok=True)
os.makedirs('./models', exist_ok=True)

print(f'Region  : {region}')
print(f'Bucket  : {bucket}')
print(f'Prefix  : {S3_PREFIX}')
print(f'Role    : {role[:80]}...')

---
## Section 1: Feature Engineering

The JumpStart LightGBM container trains on tabular CSVs 
We extract 43 lexical/structural features from each URL here, then hand the CSV to SageMaker.

| Feature group | Examples |
|---|---|
| URL-level | `url_length`, `url_entropy`, `has_https` |
| Domain | `domain_entropy`, `num_subdomains`, `has_ip_address` |
| Path / Query | `path_depth`, `num_query_params`, `has_fragment` |
| Character counts | `num_dots`, `digit_ratio`, `num_percent_signs` |
| Suspicious signals | `num_suspicious_keywords`, `has_hex_encoding`, `is_url_shortened` |
| Word-level | `num_words`, `longest_word_length`, `avg_word_length` |

In [ ]:
kagglehub.login()
path     = kagglehub.dataset_download('sid321axn/malicious-urls-dataset')
csv_file = os.path.join(path, 'malicious_phish.csv')

df = pd.read_csv(csv_file)

if SAMPLE_SIZE:
    df, _ = train_test_split(
        df, train_size=SAMPLE_SIZE, stratify=df['type'], random_state=RANDOM_SEED
    )
    df = df.reset_index(drop=True)

print(f'Dataset: {len(df):,} rows')
print(df['type'].value_counts())
print(df.head(3))

In [ ]:
print(f'Extracting {len(FEATURE_COLS_ORDER)} features from {len(df):,} URLs')
t0 = time.time()

feat_records = df['url'].progress_apply(extract_url_features)
feat_df      = pd.DataFrame(list(feat_records))[FEATURE_COLS_ORDER]

print(f'Done in {time.time() - t0:.1f}s  —  shape: {feat_df.shape}')
print('\nSample (first 3 rows):')
display(feat_df.head(3))

feat_df['type'] = df['type'].values
print('\nPer-class means (selected features):')
display(feat_df.groupby('type')[[
    'url_length', 'url_entropy', 'num_subdomains',
    'has_ip_address', 'num_suspicious_keywords', 'digit_ratio'
]].mean().round(3))
feat_df = feat_df.drop(columns=['type'])

In [ ]:
CLASS_ORDER = ['benign', 'phishing', 'defacement', 'malware']
label_map   = {cls: idx for idx, cls in enumerate(CLASS_ORDER)}

feat_df['label'] = df['type'].map(label_map).values

train_df, test_df = train_test_split(
    feat_df, test_size=0.20,
    stratify=feat_df['label'],
    random_state=RANDOM_SEED
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train : {len(train_df):,}  |  Test: {len(test_df):,}')
for name, split in [('Train', train_df), ('Test', test_df)]:
    dist = split['label'].map(dict(enumerate(CLASS_ORDER))).value_counts()
    print(f'  {name}: {dist.to_dict()}')

test_features = test_df[FEATURE_COLS_ORDER].values.astype(np.float64)
test_labels   = test_df['label'].values
test_df.to_csv('./data/test.csv', index=False)


train_jmpst = pd.concat([
    train_df[['label']],
    train_df[FEATURE_COLS_ORDER]
], axis=1)
train_jmpst.to_csv('./data/train.csv', index=False, header=False)
print(f'\nTraining CSV: {train_jmpst.shape} → ./data/train.csv  (no header row)')
print(f'Column order: label(0), {FEATURE_COLS_ORDER[0]}(1), {FEATURE_COLS_ORDER[1]}(2), ... {FEATURE_COLS_ORDER[-1]}({len(FEATURE_COLS_ORDER)})')

s3_client    = boto3.client('s3')
train_s3_key = f'{S3_PREFIX}/input/train/train.csv'
s3_client.upload_file('./data/train.csv', bucket, train_s3_key)

train_s3_uri  = f's3://{bucket}/{S3_PREFIX}/input/train/'
output_s3_uri = f's3://{bucket}/{S3_PREFIX}/output/'

print(f'\nTraining data uploaded: {train_s3_uri}')
print(f'Model output path     : {output_s3_uri}')

---
## Section 2: Amazon SageMaker JumpStart Training Job

Amazon SageMaker AI provides a managed LightGBM container via **JumpStart** (`lightgbm-classification-model`). 

In [ ]:
TRAIN_MODEL_ID      = 'lightgbm-classification-model'
TRAINING_INSTANCE   = 'ml.m5.xlarge'

lgbm_config = JumpStartConfig(
    model_id=TRAIN_MODEL_ID
)

print(f'JumpStart model ID  : {TRAIN_MODEL_ID}')
print(f'Training instance   : {TRAINING_INSTANCE}')

In [ ]:
trainer = ModelTrainer.from_jumpstart_config(
    jumpstart_config=lgbm_config,
    compute=Compute(instance_type=TRAINING_INSTANCE),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=360000),
    hyperparameters={
        "num_boost_round": 1000,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 5,
        "max_depth": -1,
        "min_child_samples": 20,
        "objective": "multiclass",
        "num_class": 4,
        "metric": "multi_logloss",
        "verbose": -1,
    },
    sagemaker_session=sess,
    role=role,
    base_job_name="phishing-lgbm",
)

print('ModelTrainer configured.')
print(f'Base job name : phishing-lgbm')
print(f'Instance type : {TRAINING_INSTANCE}')
print(f'Features      : {len(FEATURE_COLS_ORDER)} URL features + 1 label column')
print(f'Classes       : {CLASS_ORDER}')

In [ ]:
print(f'Training data : {train_s3_uri}')
print()

trainer.train(
    input_data_config=[
        InputData(channel_name="training", data_source=train_s3_uri),
    ],
    wait=True,
    logs=True,
)

print(f'\nTraining complete.')
print(f'Model artifact: {trainer._latest_training_job.model_artifacts.s3_model_artifacts}')

---
## Section 3: Deploy to Amazon SageMaker AI Real-Time Endpoint

In [ ]:
INFERENCE_INSTANCE = 'ml.m5.large'
ENDPOINT_NAME      = 'phishing-lgbm-ep'

deploy_image_uri = retrieve_image_uri(                                                                                                                                                                            
      framework=None,                                       
      region=region,                                                                                                                                                                                                
      model_id="lightgbm-classification-model",
      model_version="*",
      image_scope="inference",                                                                                                                                                                                      
      instance_type=INFERENCE_INSTANCE,                                                                                                                                                                             
  ) 

print(f'Inference image  : {deploy_image_uri}')
print(f'Endpoint name    : {ENDPOINT_NAME}')
print(f'Instance type    : {INFERENCE_INSTANCE}')
print()

model = Model.create(
    model_name=ENDPOINT_NAME + "-model",
    primary_container=ContainerDefinition(
        image=deploy_image_uri,
        model_data_url=trainer._latest_training_job.model_artifacts.s3_model_artifacts,
    ),
    execution_role_arn=role,
    region=region,
)

endpoint_config = EndpointConfig.create(
    endpoint_config_name=ENDPOINT_NAME + "-config",
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=ENDPOINT_NAME + "-model",
            initial_instance_count=1,
            instance_type=INFERENCE_INSTANCE,
            initial_variant_weight=1.0,
        )
    ],
    region=region,
)

endpoint = Endpoint.create(
    endpoint_name=ENDPOINT_NAME,
    endpoint_config_name=ENDPOINT_NAME + "-config",
    region=region,
)
endpoint.wait_for_status(target_status="InService")
print(f'Endpoint live: {ENDPOINT_NAME} Copy endpoint name to use in 03_phishing_detection_pipeline.ipynb')

---
## Section 4: Testing and Evaluation

The endpoint accepts`text/csv`rows of pre-extracted features.  
It returns **`{"probabilities": [[p_benign, p_phishing, p_defacement, p_malware], ...]}`**.

In [ ]:
sm_runtime = boto3.client('sagemaker-runtime')


def _to_csv_bytes(feat_dict_or_array):
    """Convert a single URL features dict or a 2-D feature array to CSV bytes."""
    if isinstance(feat_dict_or_array, dict):
        row = pd.DataFrame([[feat_dict_or_array[c] for c in FEATURE_COLS_ORDER]],
                           columns=FEATURE_COLS_ORDER)
    else:
        row = pd.DataFrame(feat_dict_or_array, columns=FEATURE_COLS_ORDER)
    return row.to_csv(header=False, index=False).encode('utf-8')


def predict_url(url: str, threshold: float = 0.5) -> dict:
    """Classify a raw URL string via the SageMaker endpoint."""
    feats    = extract_url_features(url)
    response = sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='text/csv',
        Body=_to_csv_bytes(feats),
    )
    probs       = json.loads(response['Body'].read())['probabilities'][0]
    mc_class    = int(np.argmax(probs))
    p_malicious = 1.0 - float(probs[0])   # P(not benign)
    return {
        'url'           : url,
        'binary'        : 'M' if p_malicious >= threshold else 'B',
        'p_malicious'   : round(p_malicious, 4),
        'multiclass'    : CLASS_ORDER[mc_class],
        'probabilities' : {cls: round(float(p), 4)
                           for cls, p in zip(CLASS_ORDER, probs)},
    }


def batch_predict(feat_matrix: np.ndarray, batch_size: int = 1500) -> np.ndarray:
    """Run endpoint inference over a feature matrix in batches. Returns (n, 4) prob array."""
    all_probs = []
    feat_df   = pd.DataFrame(feat_matrix, columns=FEATURE_COLS_ORDER)
    for i in tqdm(range(0, len(feat_df), batch_size), desc='Predicting batches'):
        batch   = feat_df.iloc[i : i + batch_size]
        resp    = sm_runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType='text/csv',
            Body=batch.to_csv(header=False, index=False).encode('utf-8'),
        )
        all_probs.extend(json.loads(resp['Body'].read())['probabilities'])
    return np.array(all_probs)

In [ ]:
sample_urls = [
    ('phishing',     'http://000m8ih.wcomhost.com/mama/adee9/index_2.html'),
    ('benign',     'https://github.com/anthropics/anthropic-sdk-python'),
    ('benign',     'https://en.wikipedia.org/wiki/Phishing'),
    ('phishing',   'http://secure-paypal-login.hackersite.ru/verify?token=abc123'),
    ('phishing',   'http://amazon-account-verify.phish-domain.com/login'),
    ('malware',    'http://192.168.1.1/payload/dropper.exe'),
    ('phishing',   'http://104.223.93.102/pc/index.php?openid.pape.max_auth_age=0&openid.return_to=https:/www.smbc.co.jp/?ref_=nav_em_hd_re_signin&openid.identity=http:/specs.openid.net/auth/2.0/identifier_select&openid.assoc_handle=jpflex&openid.mode=checkid_setup&key=a@b.c&openid.claimed_id=http:/specs.openid.net/auth/2.0/identifier_select&openid.ns=http:/specs.openid.net/auth/2.0&ref_=nav_em_hd_clc_signinhttps:/www.smbc.co.jp'),
    ('defacement', 'http://www.legit-site.org/hacked/by/unknown'),
]

print('Endpoint Inference Demo')
print('=' * 100)
print(f"{'Actual':>12}  {'Binary':>6}  {'p(Malicious)':>12}  {'Multiclass':>12}  {'Correct?':>8}  URL")
print('-' * 100)

for actual_label, url in sample_urls:
    r   = predict_url(url)
    correct = (r['binary'] == 'M') == (actual_label != 'benign')
    tick    = 'PASS' if correct else 'FAIL'
    print(f"[{actual_label:>10}]  {r['binary']:>6}  {r['p_malicious']:>12.4f}"
          f"  {r['multiclass']:>12}  [{tick:>6}]  {url[:55]}")

In [ ]:
print(f'Running batch inference on {len(test_features):,} test samples...')
probs_all   = batch_predict(test_features)      # shape: (n_test, 4)

mc_pred     = probs_all.argmax(axis=1)          # multiclass prediction
p_malicious = 1.0 - probs_all[:, 0]            # P(not benign) for binary
bin_pred    = (p_malicious >= 0.5).astype(int)
bin_true    = (test_labels != 0).astype(int)    # 0=benign → 0, rest → 1

print(f'Done. {len(probs_all):,} predictions collected.')

In [ ]:
mc_acc = accuracy_score(test_labels, mc_pred)
mc_f1  = f1_score(test_labels, mc_pred, average='macro')

print('MULTICLASS RESULTS (4-class)')
print('=' * 60)
print(f'Accuracy : {mc_acc * 100:.2f}%')
print(f'Macro F1 : {mc_f1 * 100:.2f}%')
print()
print(classification_report(test_labels, mc_pred, target_names=CLASS_ORDER))

cm_mc  = confusion_matrix(test_labels, mc_pred)
cm_pct = cm_mc.astype(float) / cm_mc.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_pct, annot=True, fmt='.1f', cmap='Blues',
    xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual',    fontsize=11)
ax.set_title('LightGBM JumpStart — Multiclass Confusion Matrix (% of row)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('./models/cm_multiclass.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
label_str  = {1: 'M', 0: 'B'}
y_true_str = pd.Series(bin_true).map(label_str)
y_pred_str = pd.Series(bin_pred).map(label_str)

cm_bin = confusion_matrix(y_true_str, y_pred_str, labels=['M', 'B'])
TP = int(cm_bin[0, 0]);  FN = int(cm_bin[0, 1])
FP = int(cm_bin[1, 0]);  TN = int(cm_bin[1, 1])

fnr = FN / (TP + FN) * 100 if (TP + FN) else 0
fpr = FP / (FP + TN) * 100 if (FP + TN) else 0
acc = (TP + TN) / (TP + TN + FP + FN) * 100
roc = roc_auc_score(bin_true, p_malicious)
f1b = f1_score(bin_true, bin_pred)
pre = TP / (TP + FP) if (TP + FP) else 0
rec = TP / (TP + FN) if (TP + FN) else 0

print('BINARY RESULTS')
print('=' * 70)
print('CONFUSION MATRIX          Predicted')
print('                      M          B')
print(f'Actual   M       {TP:>6}     {FN:>6}')
print(f'Actual   B       {FP:>6}     {TN:>6}')
print()
print('KEY PERFORMANCE METRICS')
print('=' * 70)
print(f'FALSE NEGATIVE RATE  : {fnr:.2f}%   ({FN}/{TP+FN} malicious URLs missed)')
print(f'FALSE POSITIVE RATE  : {fpr:.2f}%   ({FP}/{FP+TN} benign URLs wrongly flagged)')
print(f'True Positive Rate   : {100-fnr:.2f}%')
print(f'True Negative Rate   : {100-fpr:.2f}%')
print(f'Overall Accuracy     : {acc:.2f}%')
print(f'ROC-AUC              : {roc:.4f}')
print()
print('ADDITIONAL METRICS')
print('=' * 70)
print(f'Precision (Malicious): {pre:.4f}')
print(f'Recall    (Malicious): {rec:.4f}')
print(f'F1-Score  (Malicious): {f1b:.4f}')
print()
print(classification_report(y_true_str, y_pred_str, labels=['M', 'B'],
                            target_names=['Malicious', 'Benign']))

cm_pct_b = cm_bin.astype(float) / cm_bin.sum(axis=1, keepdims=True) * 100
fig, ax  = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm_pct_b, annot=True, fmt='.1f', cmap='Reds',
    xticklabels=['Malicious', 'Benign'],
    yticklabels=['Malicious', 'Benign'],
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual',    fontsize=11)
ax.set_title('LightGBM JumpStart — Binary Confusion Matrix (% of row)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('./models/cm_binary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []
for th in thresholds:
    pred  = (p_malicious >= th).astype(int)
    cm_t  = confusion_matrix(bin_true, pred, labels=[1, 0])
    tp_t, fn_t = int(cm_t[0, 0]), int(cm_t[0, 1])
    fp_t, tn_t = int(cm_t[1, 0]), int(cm_t[1, 1])
    rows.append({
        'threshold': round(float(th), 2),
        'FNR_%'    : round(fn_t / (tp_t + fn_t) * 100, 2) if (tp_t + fn_t) else 0,
        'FPR_%'    : round(fp_t / (fp_t + tn_t) * 100, 2) if (fp_t + tn_t) else 0,
        'Accuracy_%': round((tp_t + tn_t) / (tp_t + tn_t + fp_t + fn_t) * 100, 2),
        'F1'       : round(float(f1_score(bin_true, pred)), 4),
    })

th_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(th_df['threshold'], th_df['FNR_%'],      'r-o',  label='False Negative Rate %', markersize=4)
ax.plot(th_df['threshold'], th_df['FPR_%'],      'b-s',  label='False Positive Rate %', markersize=4)
ax.plot(th_df['threshold'], th_df['Accuracy_%'], 'g-^',  label='Accuracy %',            markersize=4)
ax.axvline(0.5, linestyle='--', color='gray', alpha=0.6, label='Default threshold (0.5)')
ax.set_xlabel('Classification Threshold', fontsize=11)
ax.set_ylabel('Rate / Accuracy (%)',      fontsize=11)
ax.set_title('Threshold vs FNR / FPR / Accuracy  —  tune for your security posture',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('./models/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Threshold sweep:')
print(th_df.to_string(index=False))

---
## Section 5: Cleanup

The endpoint incurs cost while running. Delete it when done.

In [ ]:
# endpoint.delete()
# endpoint_config.delete()
# model.delete()
# print(f'Endpoint "{ENDPOINT_NAME}" deleted.')